# 03 — Prepare & Export

Build the master states analysis table (one row per state, all metrics),
run comparison analyses, and package the sellable dataset.

**Pipeline:**
1. Parse penalty strings into clean numerics
2. Compute per-capita rates (fatalities/100k, arrests/100k)
3. Join all tables into the master states table
4. Run IID / felony / punishment profile comparisons
5. Export to CSV + Excel + Parquet + codebook

Output lands in `export/`.

In [ ]:
import sys
sys.path.insert(0, "..")

import duckdb
import pandas as pd
from src.ingest import load_config
from src.clean_quality import run_sql, save_processed
from src.prepare import (
    build_master_table,
    build_trends_table,
    compute_rates,
    iid_comparison,
    felony_comparison,
    punishment_profile_summary,
    numeric_summary,
    package_dataset,
)

cfg = load_config("../config.yaml")
con = duckdb.connect(str("../" + cfg['settings']['duckdb_file']))
print("Project:", cfg["project_name"])
print(f"Tables: {len(con.execute('SHOW TABLES').df())}")

## 1. Build master states table

In [ ]:
master = build_master_table(con)
master.head()

In [ ]:
print(f"Shape: {master.shape}")
print(f"\nColumns ({len(master.columns)}):")
for c in master.columns:
    print(f"  {c}")

In [ ]:
numeric_summary(master)

## 2. Comparisons & analysis summaries

In [ ]:
print("=== IID vs non-IID states ===")
iid_comp = iid_comparison(master)
iid_comp

In [ ]:
print("=== Felony vs always-misdemeanor states ===")
felony_comp = felony_comparison(master)
felony_comp

In [ ]:
print("=== Punishment profile summary ===")
profile_summary = punishment_profile_summary(master)
profile_summary

In [ ]:
# Quick look: top 10 states by alcohol fatality rate per 100M VMT
master.nlargest(10, "alcohol_fatality_rate_per_100m_vmt")[
    ["state_abbr", "state_name", "alcohol_fatality_rate_per_100m_vmt",
     "max_speed_limit_mph", "pct_impaired_with_prior_dwi",
     "all_offender_iid", "punishment_profile"]
]

## 3. Build trends table (2015–2020 consistent window)

In [ ]:
trends = build_trends_table(con)
print(f"Trends: {trends.shape[0]} rows, {trends['state_fips'].nunique()} states, years {trends['year'].min()}–{trends['year'].max()}")
trends.head()

## 4. Save to processed/

In [ ]:
# Prefix paths for notebook context (running from notebooks/)
import copy
cfg_nb = copy.deepcopy(cfg)
for k in ['data_raw', 'data_interim', 'data_processed', 'export', 'outputs']:
    cfg_nb['paths'][k] = '../' + cfg_nb['paths'][k]

save_processed(master, cfg_nb, "dui_master_states.parquet")
save_processed(trends, cfg_nb, "fars_trends_2015_2020.parquet")

## 5. Package for export

Writes CSV + Excel + Parquet to `export/` with a codebook.

In [ ]:
CODEBOOK = {
    "state_fips": "Two-digit FIPS code for the state.",
    "state_abbr": "Two-letter US postal abbreviation.",
    "state_name": "Full state name.",
    "region": "Census region (Northeast, Midwest, South, West).",
    "division": "Census division (e.g. Pacific, Mountain).",
    "pop_2024": "Estimated population, 2024 (Census PEP).",
    "lat": "Latitude of state centroid (Census Gazetteer).",
    "lng": "Longitude of state centroid (Census Gazetteer).",
    "alcohol_fatality_rate_per_100k": "NHTSA-imputed alcohol-impaired fatalities per 100,000 population (2024).",
    "total_fatality_rate_per_100k": "Total traffic fatalities per 100,000 population (2024).",
    "pct_traffic_deaths_alcohol": "Percent of traffic deaths that are alcohol-impaired (NHTSA imputed, 2024).",
    "dui_arrest_rate_per_100k": "DUI arrests per 100,000 reporting population (FBI UCR, 2023). Coverage varies by state.",
    "arrest_reporting_pop": "Population covered by agencies reporting DUI arrests to UCR.",
    "alcohol_impaired_fatalities_2024": "Count of alcohol-impaired driving fatalities (NHTSA statistical imputation, BAC>=.08).",
    "total_fatalities_2024": "Total traffic fatalities in the state (NHTSA, 2024).",
    "high_bac_fatalities_2024": "Fatalities involving BAC >= .15 (NHTSA imputed, 2024).",
    "checkpoints_legal": "1 if sobriety checkpoints are legal in the state, 0 otherwise.",
    "all_offender_iid": "1 if ignition interlock is required for all first-offense DUI offenders.",
    "criminal_refusal_penalty": "1 if refusing a breath test carries a separate criminal penalty.",
    "implied_consent": "1 if the state has an implied consent law.",
    "open_container_law": "1 if the state has an open container law meeting federal TEA-21 standards.",
    "bac_limit": "Legal BAC limit for standard (non-commercial, non-underage) drivers.",
    "first_offense_felony": "1 if a first DUI offense can be charged as a felony (e.g. with child in car or injury).",
    "felony_threshold": "Number of prior offenses at which DUI becomes a felony (NaN = never a felony).",
    "ethanol_per_capita_gallons_2022": "Per capita ethanol consumption in gallons (NIAAA, 2022).",
    "total_dui_arrests": "Total DUI arrests reported to FBI UCR (2023). Not all agencies report.",
    "reporting_agencies": "Number of law enforcement agencies reporting DUI arrests.",
    "total_drivers_in_fatal_crashes": "Total drivers involved in fatal crashes (FARS, 2024).",
    "drivers_suspended_revoked": "Drivers in fatal crashes whose license was suspended/revoked at time of crash.",
    "drivers_suspended_and_impaired": "Drivers who were both suspended AND impaired in fatal crashes.",
    "pct_suspended": "Percent of fatal-crash drivers who were driving on a suspended/revoked license.",
    "pct_suspended_who_impaired": "Of suspended-license drivers in fatal crashes, percent who were also impaired.",
    "fine_min_dollars": "Minimum fine for first-offense DUI (parsed from penalty schedules).",
    "fine_max_dollars": "Maximum fine for first-offense DUI.",
    "jail_min_days": "Minimum jail time in days for first-offense DUI (0 = no mandatory jail).",
    "suspension_days": "Minimum license suspension in days for first-offense DUI.",
    "iid_category": "Ignition interlock requirement category: all_offenders, repeat_only, high_bac, other.",
    "lookback_years": "Years a prior offense counts toward felony escalation (99 = lifetime).",
    "punishment_profile": "Classification: both_jail_and_fine, fine_only, mandatory_jail_only, or minimal.",
    "license_status_sufficient": "1 if state has >= 15 suspended drivers in fatal crashes (sufficient sample for analysis).",
    "pct_impaired_with_prior_dwi": "Of impaired drivers in fatal crashes with known history, pct with a prior DWI conviction.",
    "pct_all_with_prior_dwi": "Of all drivers in fatal crashes with known history, pct with a prior DWI conviction.",
    "median_crash_speed_limit": "Median posted speed limit (mph) at fatal crash locations in the state.",
    "mean_crash_speed_limit": "Mean posted speed limit (mph) at fatal crash locations.",
    "pct_crashes_high_speed": "Pct of fatal crashes on roads with speed limit >= 55 mph.",
    "max_speed_limit_mph": "Maximum posted speed limit on rural interstates (IIHS, 2026).",
    "vmt_millions_2022": "Vehicle miles traveled in millions (FHWA, 2022).",
    "fatality_rate_per_100m_vmt": "Total traffic fatalities per 100 million VMT.",
    "alcohol_fatality_rate_per_100m_vmt": "Alcohol-impaired fatalities per 100 million VMT.",
}

NOTES = """
Sources:
- NHTSA Fatality Analysis Reporting System (FARS) 2024
- NHTSA Traffic Safety Facts: State Alcohol-Impaired-Driving Estimates (2024)
- FBI Uniform Crime Report (UCR) — DUI arrests (2023)
- NIAAA Alcohol Epidemiologic Data System — per capita consumption (2022)
- NCSL DUI/DWI criminal status laws
- IIHS/GHSA/NHTSA enforcement policy compilations
- RoadLawGuide.com and AILawyer.com penalty compilations
- U.S. Census Bureau — population estimates and state geography
- IIHS Maximum Posted Speed Limits (August 2026)
- FHWA Highway Statistics 2022 — Vehicle Miles Traveled (Table VM-2)

Caveats:
- DUI arrest data (UCR): Not all agencies report. Use dui_arrest_rate_per_100k
  (based on reporting_population) rather than raw counts for cross-state comparison.
- NHTSA imputed fatalities use statistical methods to estimate BAC for untested drivers.
  These are the official published figures and differ from raw FARS coding.
- Penalty fields reflect first-offense minimums; actual sentences vary.
- States with license_status_sufficient = 0 have too few suspended-driver observations
  for reliable analysis of that metric.

Compiled by @unwelcomedata.
"""

paths = package_dataset(
    master,
    cfg_nb,
    name="dui_by_state_v1",
    codebook=CODEBOOK,
    notes=NOTES,
)
paths

## 6. Export trends table separately

In [ ]:
TRENDS_CODEBOOK = {
    "state_fips": "Two-digit FIPS code.",
    "state_abbr": "Two-letter postal abbreviation.",
    "state_name": "Full state name.",
    "year": "Calendar year (2015–2020).",
    "impaired_fatalities_any": "Alcohol-impaired fatalities (FARS raw DRUNK_DR coding, pre-2021 methodology).",
    "total_fatalities": "Total traffic fatalities for the state-year.",
    "pct_impaired": "Percent of fatalities that were alcohol-impaired.",
}

TRENDS_NOTES = """
Source: NHTSA FARS 2015–2020 (consistent DRUNK_DR field methodology).
Note: 2021+ FARS uses a different impairment coding method (drimpair.csv code 9)
and is NOT directly comparable to earlier years. This table covers only the
methodology-consistent 2015–2020 window.

Compiled by @unwelcomedata.
"""

trends_paths = package_dataset(
    trends,
    cfg_nb,
    name="dui_trends_2015_2020",
    codebook=TRENDS_CODEBOOK,
    notes=TRENDS_NOTES,
)
trends_paths

---
**Next:** open `04-viz.ipynb` to create social media charts from this dataset.